## 1. Environment Setup & Dependencies

Install the required Python packages for running the notebook. We use `wandb` for potential logging and `tqdm` for progress bars.


In [1]:
# Cell 1 — Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric umap-learn wandb networkx tqdm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00


## 2. Repository Setup

Clone the GitHub repository to the local Colab environment and add `src` to the Python path.


In [2]:
# Cell 2 — Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')


Cloning into '/content/antenna-gnn'...
remote: Enumerating objects: 220, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 220 (delta 115), reused 186 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (220/220), 4.88 MiB | 23.69 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Repo ready at /content/antenna-gnn


## 3. Drive Mount and Path Configuration

Mount Google Drive to access `RAW_DATA` and store the computed `.pt` cache files in `DATA_ROOT`. This ensures our generated artifacts are preserved beyond the lifespan of the Colab session.


In [3]:
# Cell 3 — Mount Drive and set data paths
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed', f'{DATA_ROOT}/data/processed_finetune']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')


Mounted at /content/drive
Drive mounted. DATA_ROOT=/content/drive/MyDrive/antenna_gnn


## 4. PyG Graph Construction Logic

Core function to convert a raw antenna patch pattern and its S11 spectrum into a PyTorch Geometric `Data` object. This implements node feature engineering and virtual node connectivity.


In [4]:
import torch
from torch_geometric.data import Data
import numpy as np

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    # Node features: (N*N + 1) nodes, 5 features each
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])

    # Virtual global node (index N*N): all zeros except placeholder (is_seed=-1 for virtual node)
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)

    # 4-connectivity edges
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    # Virtual node edges (connect to all metal pixels only)
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]  # virtual edge type

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    # Target
    y = torch.tensor(s11_db, dtype=torch.float).unsqueeze(0)  # (1, 201)

    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)


## 5. Dataset Processing & Caching Loop

Iterate over the fine-tuning grid sizes. To overcome Google Drive's severe I/O latency when sequentially accessing thousands of files, we bulk-copy the raw `.mat` files to the local Colab disk using parallel workers before processing them.


In [6]:
import glob, os, torch, shutil, concurrent.futures
import scipy.io as sio
import numpy as np
from tqdm.auto import tqdm

for N in [35, 45, 55]:
    raw_files = sorted(glob.glob(f'{RAW_DATA}/fine-tuning dataset/{N}x{N}/**/Mat_Files/*.mat', recursive=True))
    seed_mask = np.load(f'{DATA_ROOT}/artifacts/seed_mask_{N}.npy')

    proc_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
    os.makedirs(proc_dir, exist_ok=True)

    print(f"Processing {N}x{N} grid, {len(raw_files)} files...")

    local_dir = f'/content/raw_{N}x{N}'
    os.makedirs(local_dir, exist_ok=True)

    print('Bulk copying files to local disk...')
    def copy_file(src):
        dst = os.path.join(local_dir, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        return dst

    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
        local_paths = list(tqdm(executor.map(copy_file, raw_files), total=len(raw_files), desc=f'Copying {N}x{N} files'))

    functioning_count = 0
    for i, local_f in enumerate(tqdm(local_paths, desc=f'Processing {N}x{N} graphs')):
        proc_path = f'{proc_dir}/sample_{i}.pt'

        if os.path.exists(proc_path):
            data = torch.load(proc_path, weights_only=False)
            functioning_count += getattr(data, 'is_functioning', 0)
        else:
            mat = sio.loadmat(local_f)
            is_functioning = int(mat['resonant_freqs'].size > 0)
            functioning_count += is_functioning

            data = build_pyg_graph(mat['patch_pattern'], mat['S11_dB'].flatten(), seed_mask, N)
            data.grid_size = N
            data.pixel_size_mm = 32.375 / N
            data.is_functioning = is_functioning
            torch.save(data, proc_path)

    print(f'Cleaning up local cache {local_dir}...')
    shutil.rmtree(local_dir)

    with open(f'{DATA_ROOT}/data/processed_finetune/{N}x{N}_DONE.txt', 'w') as fh:
        fh.write('DONE\n')

    print(f"Grid {N}x{N}: {len(raw_files)} total, {functioning_count/len(raw_files)*100:.1f}% functioning")

Processing 35x35 grid, 4988 files...
Bulk copying files to local disk...


Copying 35x35 files:   0%|          | 0/4988 [00:00<?, ?it/s]

Processing 35x35 graphs:   0%|          | 0/4988 [00:00<?, ?it/s]

Cleaning up local cache /content/raw_35x35...
Grid 35x35: 4988 total, 63.4% functioning
Processing 45x45 grid, 6984 files...
Bulk copying files to local disk...


Copying 45x45 files:   0%|          | 0/6984 [00:00<?, ?it/s]

Processing 45x45 graphs:   0%|          | 0/6984 [00:00<?, ?it/s]

Cleaning up local cache /content/raw_45x45...
Grid 45x45: 6984 total, 65.3% functioning
Processing 55x55 grid, 2992 files...
Bulk copying files to local disk...


Copying 55x55 files:   0%|          | 0/2992 [00:00<?, ?it/s]

Processing 55x55 graphs:   0%|          | 0/2992 [00:00<?, ?it/s]

Cleaning up local cache /content/raw_55x55...
Grid 55x55: 2992 total, 48.4% functioning


## 6. Stratified Data Splitting

Combine sample indices across all grids and perform a stratified 80/20 split based on `grid_size` and `is_functioning` to create the fine-tuning pool and held-out test set.


In [ ]:
# Cell 11 - Inspect Chunk 5's split structure
import json
from collections import defaultdict

chunk5_split_file = f'{DATA_ROOT}/splits/indices.json'
with open(chunk5_split_file, 'r') as f:
    c5_splits = json.load(f)

print("Top-level keys:", list(c5_splits.keys()))

grid_counts = defaultdict(lambda: defaultdict(int))

for key, entries in c5_splits.items():
    print(f"\n--- Key: {key} ---")
    print(f"Sample entries: {entries[:3]}")
    for entry in entries:
        grid_size = entry[0]
        grid_counts[grid_size][key] += 1

print("\n--- Summary per Fine-Tune Grid ---")
for N in [35, 45, 55]:
    train_count = grid_counts[N].get('train', 0)
    val_count = grid_counts[N].get('val', 0)
    test_count = grid_counts[N].get('test', 0)
    print(f"Grid {N}: train={train_count}, val={val_count}, test={test_count}")


In [ ]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

manifest_path = f'{DATA_ROOT}/artifacts/finetune_manifest.csv'

if os.path.exists(manifest_path):
    print("Loading existing manifest...")
    manifest = pd.read_csv(manifest_path)
else:
    print("Building manifest from .pt files...")
    records = []
    for N in [35, 45, 55]:
        proc_dir = f'{DATA_ROOT}/data/processed_finetune/{N}x{N}'
        pt_files = sorted(glob.glob(f'{proc_dir}/sample_*.pt'))
        
        for pt_file in tqdm(pt_files, desc=f"Scanning {N}x{N}"):
            idx = int(os.path.basename(pt_file).split('_')[1].split('.')[0])
            data = torch.load(pt_file, weights_only=False)
            records.append({
                'grid_size': N,
                'sample_idx': idx,
                'is_functioning': data.is_functioning,
                'pixel_size_mm': data.pixel_size_mm
            })
    manifest = pd.DataFrame(records)
    manifest.to_csv(manifest_path, index=False)
    print(f"Saved manifest to {manifest_path}")

pool_list = list(zip(manifest['grid_size'], manifest['sample_idx']))
labels_for_stratify = [f"{g}_{f}" for g, f in zip(manifest['grid_size'], manifest['is_functioning'])]

import os
import shutil

manifest_indexed = manifest.set_index(['grid_size', 'sample_idx'])

# Backup current split
for split_name in ['finetune_test_indices.json', 'finetune_val_indices.json', 'finetune_pool_indices.json']:
    src_path = f'{DATA_ROOT}/splits/{split_name}'
    dst_path = f'{DATA_ROOT}/splits/{split_name.replace(".json", "_legacy_independent_split.json")}'
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.copy(src_path, dst_path)
        print(f"Backed up {split_name} to legacy independent split.")

chunk5_split_file = f'{DATA_ROOT}/splits/indices.json'
with open(chunk5_split_file, 'r') as f:
    c5_splits = json.load(f)

has_val_test = any(entry[0] in [35, 45, 55] for entry in c5_splits.get('val', [])) or \
               any(entry[0] in [35, 45, 55] for entry in c5_splits.get('test', []))

if has_val_test:
    print("Case A: Using existing train/val/test splits for fine-tune grids from Chunk 5.")
    train_pool = [x for x in c5_splits.get('train', []) if x[0] in [35, 45, 55]]
    val_set = [x for x in c5_splits.get('val', []) if x[0] in [35, 45, 55]]
    test_set = [x for x in c5_splits.get('test', []) if x[0] in [35, 45, 55]]
else:
    print("Case B: Chunk 5 only has train partition for fine-tune grids. Deriving val/test...")
    c5_train_ft = set(tuple(x) for x in c5_splits.get('train', []) if x[0] in [35, 45, 55])
    
    test_set = [x for x in pool_list if tuple(x) not in c5_train_ft]
    c5_train_list = [x for x in pool_list if tuple(x) in c5_train_ft]
    
    c5_train_labels = [f"{x[0]}_{manifest_indexed.loc[(x[0], x[1]), 'is_functioning']}" for x in c5_train_list]
    
    train_pool, val_set = train_test_split(
        c5_train_list, test_size=0.15, stratify=c5_train_labels, random_state=42
    )

with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'w') as f:
    json.dump(test_set, f)
with open(f'{DATA_ROOT}/splits/finetune_val_indices.json', 'w') as f:
    json.dump(val_set, f)
with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'w') as f:
    json.dump(train_pool, f)

assert len(train_pool) + len(val_set) + len(test_set) == len(manifest), "Union of splits does not equal full manifest!"
set_train = set(tuple(x) for x in train_pool)
set_val = set(tuple(x) for x in val_set)
set_test = set(tuple(x) for x in test_set)
assert set_train.isdisjoint(set_val)
assert set_val.isdisjoint(set_test)
assert set_train.isdisjoint(set_test)

for name, dataset in [("Pool", train_pool), ("Val", val_set), ("Test", test_set)]:
    print(f"\n{name} set breakdown:")
    for N in [35, 45, 55]:
        grid_items = [x for x in dataset if x[0] == N]
        count = len(grid_items)
        if count > 0:
            func = sum(manifest_indexed.loc[(N, idx), 'is_functioning'] for _, idx in grid_items)
        else:
            func = 0
        print(f"  {N}x{N}: {count} (Func: {func})")


## 7. Normalization Statistics

### Why Reuse Normalization Statistics?
It is **CRITICAL** to use the exact same `s11_mean.npy` and `s11_std.npy` computed from the **COMBINED multi-grid training split (25x25 + 35/45/55) in Chunk 5**. Do **NOT** recompute new normalization stats from this fine-tuning dataset independently. The pretrained model's output layer expects targets normalized under the original training distribution. Changing the normalization here would invalidate the pretrained weights and silently break the transfer learning initialization.


In [9]:
import numpy as np

# Load s11_mean and s11_std computed from the COMBINED multi-grid training split (Chunk 5)
s11_mean = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
s11_std  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')

print(f"s11_mean shape: {s11_mean.shape}")
print(f"s11_std shape: {s11_std.shape}")


s11_mean shape: (201,)
s11_std shape: (201,)


In [ ]:
import json
import os
import numpy as np

print("Files in DATA_ROOT/splits/:")
splits_dir = f'{DATA_ROOT}/splits'
print(os.listdir(splits_dir))

chunk5_split_file = f'{splits_dir}/indices.json'
with open(chunk5_split_file, 'r') as f:
    c5_splits = json.load(f)

c5_train = c5_splits['train']
# Extract (grid_size, sample_idx) pairs for 35/45/55
c5_ft_train_set = set(tuple(x) for x in c5_train if x[0] in [35, 45, 55])

# Load finetune test indices
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'r') as f:
    test_indices = json.load(f)

test_set_tuples = set(tuple(x) for x in test_indices)
intersection = c5_ft_train_set.intersection(test_set_tuples)

print(f"Number of test samples that contributed to norm stats (Chunk 5 train): {len(intersection)}")
print(f"Percentage of test set: {len(intersection) / max(len(test_set_tuples), 1) * 100:.2f}%")

assert len(intersection) == 0, "Leakage persists after adopting the Chunk 5 split — investigate before proceeding to Chunk 13."

print("\nNote: The ORIGINAL s11_mean.npy and s11_std.npy remain the canonical stats for all downstream chunks,")
print("for consistency with 25x25 pretraining. The model head is calibrated to these stats.")

# Diagnostic only, for the paper's methods section
try:
    s11_mean_clean = np.load(f'{DATA_ROOT}/artifacts/s11_mean_clean.npy')
    s11_std_clean = np.load(f'{DATA_ROOT}/artifacts/s11_std_clean.npy')
    s11_mean = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
    s11_std = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')

    std_diff = np.abs(s11_std.squeeze() - s11_std_clean.squeeze())
    s11_std_sq = s11_std.squeeze()
    s11_std_clean_sq = s11_std_clean.squeeze()
    
    rel_diff = std_diff / s11_std_sq
    max_rel_diff = np.max(rel_diff)
    max_idx = np.argmax(std_diff)
    
    print("\n--- Diagnostic for Methods Section ---")
    print(f"Max relative difference in std: {max_rel_diff:.6%}")
    print(f"Absolute max difference occurred at frequency index {max_idx}.")
    print(f"Original std at index {max_idx}: {s11_std_sq[max_idx]:.6f}")
    print(f"Clean std at index {max_idx}: {s11_std_clean_sq[max_idx]:.6f}")

except FileNotFoundError:
    print("\nDiagnostic step skipped: _clean stats not found (no previous leakage detected).")


## 8. Verification & DataLoader Test

Verify that the DataLoader outputs properly batched graph objects with mixed grid sizes, and spot check the fixed test set.


In [11]:
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader

class FinetuneDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        grid_size, local_idx = self.indices[idx]
        return torch.load(f'{DATA_ROOT}/data/processed_finetune/{grid_size}x{grid_size}/sample_{local_idx}.pt', weights_only=False)

with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'r') as f:
    pool_indices = json.load(f)

pool_dataset = FinetuneDataset(pool_indices)
loader = DataLoader(pool_dataset, batch_size=16, shuffle=True)

batch = next(iter(loader))
print("Batch from Pool:")
print(batch)
print("Grid sizes in this batch:", batch.grid_size.tolist())
print("NaNs in x:", torch.isnan(batch.x).any().item())
print("NaNs in y:", torch.isnan(batch.y).any().item())

print("\nSpot check FIXED test set:")
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'r') as f:
    test_indices = json.load(f)

test_dataset = FinetuneDataset(test_indices)
for i in range(5):
    sample = test_dataset[i]
    print(f"Sample {i}: grid_size={sample.grid_size}, is_functioning={sample.is_functioning}")

Batch from Pool:
DataBatch(x=[29616, 5], edge_index=[2, 153698], edge_attr=[153698, 2], y=[16, 201], grid_size=[16], pixel_size_mm=[16], is_functioning=[16], batch=[29616], ptr=[17])
Grid sizes in this batch: [45, 45, 35, 35, 45, 45, 45, 35, 45, 45, 35, 55, 35, 55, 35, 45]
NaNs in x: False
NaNs in y: False

Spot check FIXED test set:
Sample 0: grid_size=45, is_functioning=0
Sample 1: grid_size=45, is_functioning=1
Sample 2: grid_size=35, is_functioning=1
Sample 3: grid_size=55, is_functioning=1
Sample 4: grid_size=45, is_functioning=1


In [ ]:
import inspect
import random

# Assert build_pyg_graph source contains exactly the virtual node definition
source_code = inspect.getsource(build_pyg_graph)
assert "[0.0, 0.5, 0.5, -1.0, 0.0]" in source_code, "build_pyg_graph virtual node definition modified or missing 'is_seed=-1' semantics!"

# Sample a random processed .pt file from the test set to verify shapes
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json', 'r') as f:
    test_indices = json.load(f)

sample = random.choice(test_indices)
grid_size, local_idx = sample
pt_path = f'{DATA_ROOT}/data/processed_finetune/{grid_size}x{grid_size}/sample_{local_idx}.pt'
data = torch.load(pt_path, weights_only=False)

assert data.x.shape[1] == 5, f"Expected 5 node features, got {data.x.shape[1]}"
assert data.edge_attr.shape[1] == 2, f"Expected 2 edge features, got {data.edge_attr.shape[1]}"
assert data.y.shape == (1, 201), f"Expected y.shape == (1, 201), got {data.y.shape}"
assert data.y.min() < 0, f"Expected min(y) < 0 (raw dB), got {data.y.min()}"

# Assert splits are disjoint
val_indices = json.load(open(f'{DATA_ROOT}/splits/finetune_val_indices.json', 'r'))
pool_indices = json.load(open(f'{DATA_ROOT}/splits/finetune_pool_indices.json', 'r'))

set_test = set(tuple(x) for x in test_indices)
set_val = set(tuple(x) for x in val_indices)
set_pool = set(tuple(x) for x in pool_indices)

assert set_test.isdisjoint(set_val), "Test and Val splits overlap!"
assert set_val.isdisjoint(set_pool), "Val and Pool splits overlap!"
assert set_test.isdisjoint(set_pool), "Test and Pool splits overlap!"
print("Guard cell assertions passed.")

